# Adjoint capture analysis: streamflow and lake capture

When a new well starts pumping, the water it produces has to come from
somewhere. Some of it is released from aquifer storage, and the rest is
**capture** — water that would otherwise have discharged to a stream or a lake,
or that the stream and lake now lose to the aquifer. This notebook computes
streamflow capture and lake capture for the synthetic-valley model using
**adjoint-state sensitivity analysis** with
[mf6adj](https://github.com/INTERA-Inc/mf6adj).

By the end of this notebook you will be able to:

- write an mf6adj **performance measure** for the flow through a boundary package,
- run a forward and an adjoint solve and read the resulting sensitivities,
- turn the sensitivity of a stream or lake flux to a well rate into a **capture
  fraction**,
- map where in the aquifer that capture is most sensitive to hydraulic
  conductivity, and
- check the adjoint result against the brute-force answer from two model runs.

## What an adjoint sensitivity is

The direct way to find out how sensitive a model result is to a parameter is to
change the parameter and run the model again. That costs one model run per
parameter, so a sensitivity map over a 5-layer, 40-by-25 grid would cost
thousands of runs.

The adjoint approach turns the problem around. You first name the single model
output you care about — the **performance measure** — and then solve the flow
equations *backward* in time once. That one backward solve returns the
sensitivity of that measure to every parameter in every cell: hydraulic
conductivity, storage, recharge, boundary conductance, and well rates.

The trade-off is the mirror image of the direct approach: one backward solve per
measure, rather than one forward run per parameter. mf6adj drives MODFLOW 6
through its Application Programming Interface (**API**) to collect the matrices
it needs, so the model itself is unmodified.

The method and its verification are described in Hayek and others (2025),
*MF6-ADJ: A Non-Intrusive Adjoint Sensitivity Capability for MODFLOW 6*,
Groundwater 63(6), 874–888.

Import the packages this notebook uses. `mf6adj` provides the adjoint solver and
`mf6_adj_helpers` collects the workspace setup shared by the two adjoint
notebooks.

In [ ]:
import flopy
import matplotlib.pyplot as plt
import mf6_adj_helpers as adjh
import mf6adj
import numpy as np
from mf6_notebook_helpers import find_mf6_libraries

Locate the MODFLOW 6 shared library and executable in the active environment.
mf6adj drives the model through the shared library (`libmf6`), so both paths are
needed: the executable for the ordinary forward runs and the library for the
adjoint.

In [ ]:
lib_name, mf6_exe = find_mf6_libraries()
print(f"library:    {lib_name.name}")
print(f"executable: {mf6_exe.name}")

## Prepare the model

Use the **advanced** synthetic-valley model at an annual sampling frequency. It
represents the valley with the advanced hydrologic packages: streamflow routing
(**SFR**) for the river, a lake (**LAK**), unsaturated-zone flow (**UZF**) for
recharge and evapotranspiration, and the water mover (**MVR**) to route water
between them. The 21 stress periods start with a long steady spin-up followed by
20 annual periods.

One change is needed first. The production wells in the advanced model are
multi-aquifer wells (**MAW**), and MAW adds one equation per well to the
MODFLOW 6 solution matrix. mf6adj rebuilds the adjoint matrix from the
groundwater-flow grid connectivity alone, so it cannot use a solution matrix
that carries extra equations, and it stops with an error. `prepare_model()`
swaps the MAW package for the equivalent WEL cells that ship with the model —
the same two wells at the same rates, split across layers 4 and 5. SFR, LAK, and
UZF need no such change: they are solved in the outer (Picard) iteration and add
nothing to the matrix.

In [ ]:
ws = adjh.prepare_model("adj-capture", variant="advanced", prediction_rate=0.0)
adjh.run_model(ws, mf6_exe)

sim = flopy.mf6.MFSimulation.load(sim_ws=str(ws), verbosity_level=0)
gwf = sim.get_model()
nper = sim.tdis.nper.data
print(f"stress periods: {nper}")
print(f"packages:       {', '.join(sorted(gwf.package_names))}")

This first run is the **baseline**: the prediction well is present but pumps at a
rate of zero. The sensitivities describe how the model responds to a small change
away from this baseline, which is what makes the capture fraction a property of
the aquifer and the well's position rather than of one particular pumping rate.

### Look at the model

Map the features that matter for capture. Plot the lake cells and the stream
reaches with `.plot_bc()`, and mark the two production wells and the prediction
well. The prediction well sits in the southern part of the valley, near the
stream and far from the lake.

In [ ]:
pred_cell = (4, 34, 15)  # zero-based (layer, row, column)
prod_cells = adjh.package_cells(gwf, "pwell")

fig, ax = plt.subplots(figsize=(6, 8), constrained_layout=True)
mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
mm.plot_grid(lw=0.2, color="0.8")
mm.plot_bc("LAK", color="tab:blue")
mm.plot_bc("SFR", color="tab:cyan")
mm.plot_ibound()

xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
for k, i, j in prod_cells:
    ax.plot(xc[i, j], yc[i, j], "ko", ms=7)
ax.plot(
    xc[pred_cell[1], pred_cell[2]],
    yc[pred_cell[1], pred_cell[2]],
    "r*",
    ms=16,
    label="prediction well",
)
ax.plot([], [], "ko", ms=7, label="production wells")
ax.plot([], [], "s", color="tab:blue", label="lake")
ax.plot([], [], "s", color="tab:cyan", label="stream")
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Synthetic valley: surface-water features and wells")

**What to look for.** The lake occupies the northern third of the valley and the
stream runs the length of it. The prediction well (red star) is in the south,
close to the stream and roughly 20 miles from the lake. Expect the stream to
supply most of the captured water and the lake very little.

## Define the performance measures

An mf6adj performance measure is a plain-text block listing the model outputs to
combine. Each entry is one line:

```
# kper kstp layer row column  package  form  weight  observed
  21    1     3    18    5    sfr-1   direct  1.0    -1.0e+30
```

The indices are **one-based** in this file, unlike the zero-based indices used
everywhere in FloPy. The `package` field is either `head` — for the head in that
cell — or the name of a boundary package from the model name file, which selects
the flow between that package and the cell. `direct` means the measure is the
value itself rather than a residual against an observation, so the `observed`
field is unused.

Build two measures at the last stress period: `swgw` sums the exchange between
the stream and the aquifer over all 18 stream reaches, and `lakegw` sums the
exchange between the lake and the aquifer over all 105 lake connections.
Together they are the total surface-water flow that pumping can capture.

In [ ]:
last = nper - 1  # zero-based
sfr_cells = adjh.package_cells(gwf, "sfr-1")
lak_cells = adjh.package_cells(gwf, "lak-1")
print(f"stream reaches:   {len(sfr_cells)}")
print(f"lake connections: {len(lak_cells)}")

measures = {
    "swgw": [(last, 0, k, i, j, "sfr-1") for k, i, j in sfr_cells],
    "lakegw": [(last, 0, k, i, j, "lak-1") for k, i, j in lak_cells],
}
adj_file = adjh.write_adj_file(ws, "capture.adj", measures)
print(adj_file.read_text()[:260] + "...")

## Solve the forward and adjoint problems

Create the `mf6adj.Mf6Adj` object with the measure file and the shared library.
`solve_forward_model()` runs MODFLOW 6 through the API and saves the solution
matrix and boundary terms at every time step; `solve_adjoint()` then sweeps
backward through those saved time steps, once per measure. Always call
`finalize()` to release the library. Set `logging_level="WARNING"` to keep the
per-time-step progress messages out of the notebook.

`WARNING` is the level to keep, though, because mf6adj reports there when part
of a package's derivative is not formed. This model draws two, both about the
lake, and they matter for how you read the lake result below.


In [ ]:
adj = mf6adj.Mf6Adj(
    adj_file.name,
    str(lib_name),
    logging_level="WARNING",
    working_directory=str(ws),
)
adj.solve_forward_model()
sensitivities = adj.solve_adjoint()
adj.finalize()

print(f"measures solved: {', '.join(sensitivities)}")
print(f"parameters:      {', '.join(sensitivities['swgw'].columns)}")

**What to look for.** Two warnings name the parts of the lake derivative
mf6adj does not form: the inflow the water mover (**MVR**) routes into the lake,
which follows the state of the package supplying it, and the lake's horizontal
connections, whose conductance and wetted area follow the stage through the
saturated fraction of the cell. Neither is differentiated, so the lake
sensitivity is approximate and mf6adj says so rather than leaving you to find
out. The stream draws no such warning.


Each measure returns a table with one row per model cell and one column per
parameter — hydraulic conductivity (`k11`, `k33`), the two storage properties
(specific storage `ss` and specific yield `sy`), recharge, the stream and lake
stage and conductance, and the well rate (`wel6_q`). Every one of those columns
came from the one backward solve.

## Read the capture fractions

The sensitivity of a stream or lake flux to a well rate *is* the capture
fraction: it is the change in surface-water flow per unit of pumping, so a value
of -1 means every unit pumped is taken from that feature and 0 means none of it
is.

Read the `wel6_q` sensitivity at the prediction well cell. Sum the per-period
sensitivities over the periods the well actually pumps (12 through 21), because
each period's value is that period's contribution to the measure at the final
period.

In [ ]:
active = range(11, nper)  # zero-based periods the prediction well pumps
capture = {}
for measure in ("swgw", "lakegw"):
    capture[measure] = adjh.total_sensitivity(
        ws, measure, "wel6_q", cell=pred_cell, periods=active
    )

print(f"streamflow capture fraction: {capture['swgw']:8.4f}")
print(f"lake capture fraction:       {capture['lakegw']:8.4f}")
print(f"total surface-water capture: {sum(capture.values()):8.4f}")

**What to look for.** Roughly half of the water pumped by the prediction well is
taken from the stream, and the lake contributes almost nothing — under one
percent — as the map suggested. The remainder comes from aquifer storage and
from reduced evapotranspiration. The signs are negative because the well rate is
itself negative: pumping harder makes the stream give up more water.

### How capture builds up through time

The per-period sensitivities of a measure defined at the *last* period show how
far back in time the measure remembers. Plot the stream capture contributed by
each stress period.

In [ ]:
per_period = adjh.period_sensitivity(ws, "swgw", "wel6_q")
contrib = np.array([per_period[kper][pred_cell] for kper in sorted(per_period)])

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.bar(np.arange(1, nper + 1), -contrib, color="tab:cyan", edgecolor="k", lw=0.4)
ax.set_xlabel("stress period")
ax.set_ylabel("contribution to streamflow capture (-)")
ax.set_title("Where the final-period capture comes from in time")
ax.set_xticks(range(2, nper + 1, 2))

**What to look for.** The last stress period dominates: pumping in the year the
measure is evaluated does most of the work. Earlier years contribute a rapidly
shrinking amount, which is the aquifer's memory of past pumping. The measure
still depends on periods before the well turned on, but by a negligible amount.

## Map where capture is most sensitive

The same backward solve produced the sensitivity of stream capture to hydraulic
conductivity in every cell. Map the composite `k11` sensitivity for the top
layer to see which parts of the aquifer control how much of the pumping the
stream ends up supplying.

In [ ]:
k11_sens = adjh.composite_sensitivity(ws, "swgw", "k11")

fig, ax = plt.subplots(figsize=(6, 8), constrained_layout=True)
mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
vmax = np.percentile(np.abs(k11_sens[0]), 99)
cb = mm.plot_array(k11_sens[0], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
mm.plot_bc("SFR", color="tab:cyan")
mm.plot_ibound()
ax.plot(xc[pred_cell[1], pred_cell[2]], yc[pred_cell[1], pred_cell[2]], "k*", ms=14)
plt.colorbar(cb, ax=ax, shrink=0.5, label="d(stream flux) / d(k11)")
ax.set_title("Sensitivity of streamflow capture to layer-1 conductivity")

**What to look for.** The sensitivity is concentrated in a band between the well
and the stream: those are the cells whose conductivity controls how easily water
moves from the stream toward the well. Cells far from that path barely matter,
however uncertain their conductivity is. This is the practical value of an
adjoint — it says where to spend the next field measurement.

## Check the adjoint against two model runs

The brute-force way to get a capture fraction is to run the model twice, once
with the well off and once with it on, and difference the budgets. That gives
one number per run, but it is an independent check on the adjoint.

Run the model again with the prediction well pumping at a small rate. Keep the
rate small: the adjoint returns a derivative evaluated at zero pumping, so the
difference only matches it in the limit of a small perturbation. Tighten the
solver at the same time, because the differences in lake flow are small enough
to be lost in ordinary convergence noise.

In [ ]:
dq = -3000.0  # ft^3/d, small enough to stay in the linear range

ws_base = adjh.prepare_model(
    "adj-capture-q0", prediction_rate=0.0, outer_dvclose=1.0e-10
)
ws_pert = adjh.prepare_model(
    "adj-capture-dq", prediction_rate=dq, outer_dvclose=1.0e-10
)
for w in (ws_base, ws_pert):
    adjh.run_model(w, mf6_exe)

print("               adjoint   two-run difference")
for measure, term in (("swgw", "SFR(SFR-1)"), ("lakegw", "LAK(LAK-1)")):
    diff = (
        adjh.budget_net(ws_pert, term)[last] - adjh.budget_net(ws_base, term)[last]
    ) / dq
    print(f"  {measure:8s} {capture[measure]:10.5f} {diff:15.5f}")

**What to look for.** The stream measure agrees closely — the adjoint and the
two-run difference land within about one percent of each other, which verifies
the whole chain from the saved matrices to the capture fraction.

The lake measure still does not agree, but the reason is narrower than it used
to be. mf6adj now solves the lake's water balance along with the flow equations,
so the lake surface is free to rise and fall in response to the pumping instead
of being pinned where it started. The sensitivity it returns therefore includes
the knock-on effect: pumping lowers the heads beneath the lake, the lake gives up
water, its surface drops, and that drop in turn slows the leakage. Holding the
surface still, as it used to, left that feedback out.

What is still left out is named in the warnings above — the inflow the mover
routes into the lake, and the lake's horizontal connections. Both are used in
this model, so the lake number remains approximate, and here what is missing is
large enough compared with the number itself to flip its sign.

Read the numbers with their size in mind. The lake terms are all under half a
percent of the stream's, so the practical conclusion — this well captures water
from the stream, not the lake — is the same whichever you trust. The lesson is
that an adjoint sensitivity answers exactly the question its formulation
encodes, and that mf6adj now tells you which parts of that formulation are
approximate. Where the distinction matters, the two-run difference is the number
to check against.


## Recap

- A **performance measure** names the model output you care about; one backward
  adjoint solve returns its sensitivity to every parameter in every cell.
- The sensitivity of a stream or lake flux to a well rate is the **capture
  fraction** directly.
- For the prediction well, roughly half the pumped water is captured from the
  stream and almost none from the lake.
- The per-period sensitivities show the aquifer's memory: recent pumping
  dominates the capture at any given time.
- Mapping the sensitivity to hydraulic conductivity shows *where* the aquifer
  controls capture, which is where better data would be worth collecting.
- mf6adj warns when part of a package's derivative is not formed. Read those
  warnings, and check a measure against a small-perturbation two-run difference
  when they name something the model actually uses — as the lake's mover inflow
  and horizontal connections are used here.
